# Stage 3 — DFC, one dataset per cohort

Windowed connectivity: one row per window, twelve QC columns, and the edges —
stored one column per edge below ~20,000 of them and as a packed
`fixed_size_list<float32>` above.

Same four steps as `01_activation.ipynb`:

1. **inventory** the `(atlas, window_s, cohort)` grid that actually exists;
2. **load QC first** — twelve columns, a few MB per cohort, on any atlas —
   because that is what tells you which windows are worth loading edges for;
3. **edges**, both storage modes, per subject on a fine atlas and pooled on a
   coarse one;
4. **join** participants, QC and phenotype.

The edge columns are the memory cliff in this project, so nothing here
materialises them without pricing the read first.

### Running it

```bash
module load apptainer/1.4.5
salloc --account=rpp-aevans-ab --cpus-per-task=4 --mem=32G --time=3:00:00

export FMRIDECOMP_SIF=/project/6008063/tamires/singularity/fmri_decomp.sif
cd /project/6008063/tamires/DecomposingfMRI
apptainer exec --bind /project,/scratch,/home --pwd $PWD $FMRIDECOMP_SIF \
    jupyter lab --no-browser --ip=0.0.0.0 --port=8888
```

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.dataset as pads

sys.path.insert(0, str(Path.cwd()))
import nbtools as nb

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 60)

try:
    import matplotlib.pyplot as plt
    HAVE_MPL = True
except ImportError:
    HAVE_MPL = False
    print("matplotlib absent -- plots are skipped, tables still print")

ATLAS_COARSE = "yeo7"            # 21 edges: a whole cohort of edges is cheap
ATLAS_FINE = "harvardoxford"     # 6,105 edges: per subject, or QC only
WINDOW_S = 60                    # one aperture, present in every cohort's grid

ROOT = nb.output_root()
print("output_root:", ROOT)

## 1. Which `(atlas, window_s, cohort)` cells exist

`window_s` sits between `atlas` and `cohort` in the path, so "one atlas, one
window size, every cohort" is a single directory — the pooled read. The grid is
deliberately not full: `windows.by_size` restricts the 15 s aperture to the
coarse atlases, because a correlation over *p* nodes is singular unless
`n − 1 ≥ p` and a 15 s window is 10 samples at TR = 1.49.

In [ ]:
inv = nb.inventory("dfc")
print(f"{len(inv)} dfc shard(s)")
nb.inventory_summary(inv, "dfc").head(20)

In [ ]:
# The grid, as shards. A zero is a pair that was never run -- check it against
# `windows.sizes_s` and `windows.by_size` in the config before calling it a gap.
inv.assign(window_s=inv["window_s"].astype(float)).pivot_table(
    index=["cohort", "atlas"], columns="window_s", values="path",
    aggfunc="size", fill_value=0)

In [ ]:
# Configured vs. present, per cohort. `configured` is `windows.sizes_s`; a size
# that is configured but absent for one atlas only is the by_size restriction,
# not a failure.
cfg = nb.cohort_configs().set_index("cohort")
rows = []
for cohort in sorted(inv["cohort"].unique()):
    sub = inv[inv["cohort"] == cohort]
    configured = {float(w) for w in str(cfg.loc[cohort, "window_sizes_s"]).split(",") if w}
    present = {float(w) for w in sub["window_s"].unique()}
    rows.append({"cohort": cohort,
                 "configured": sorted(configured),
                 "present": sorted(present),
                 "absent": sorted(configured - present),
                 "atlases": sorted(sub["atlas"].unique())})
pd.DataFrame(rows)

### The rank floor is arithmetic, not a preference

A window is `round(window_s / tr)` samples; a correlation matrix over *p* nodes
is singular unless `n − 1 ≥ p`. So the shortest usable window is a property of
the atlas and the TR:

```
min_window_s = (n_nodes + 0.5) * tr
```

The pipeline computes edges below that floor anyway and flags the rows
`rank_deficient` (`rank_policy: warn`) rather than dropping them — so the
correct number of rank-deficient windows is *predictable*, and a disagreement
between prediction and data is worth chasing.

In [ ]:
n_nodes = {a: len(nb.atlas_labels(a)) for a in sorted(inv["atlas"].unique())}
print("nodes per atlas:", n_nodes)

grid = []
for cohort in sorted(inv["cohort"].unique()):
    tr = float(cfg.loc[cohort, "tr"])
    for atlas, p in n_nodes.items():
        floor_s = (p + 0.5) * tr
        for w in sorted({float(x) for x in
                         inv.loc[(inv["cohort"] == cohort) & (inv["atlas"] == atlas),
                                 "window_s"]}):
            grid.append({"cohort": cohort, "tr": tr, "atlas": atlas, "n_nodes": p,
                         "window_s": w, "n_samples": int(round(w / tr)),
                         "min_window_s": round(floor_s, 1),
                         "expect_rank_deficient": w < floor_s})
grid = pd.DataFrame(grid)
grid[grid["expect_rank_deficient"]].sort_values(["cohort", "atlas", "window_s"])

## 2. QC first

Twelve columns. Parquet is columnar, so this physically reads twelve column
chunks per file and skips the edges entirely — the same read costs the same
whether the atlas has 21 edges or 6,105. This is always the first read.

In [ ]:
qc_ds, _ = nb.dataset("dfc", ATLAS_FINE)
filt = pads.field("window_s") == str(WINDOW_S)
gb_qc, n_frag = nb.estimate_gb(qc_ds, columns=list(nb.DFC_QC_COLUMNS), filter=filt)
gb_all, _ = nb.estimate_gb(qc_ds, filter=filt)
print(f"{ATLAS_FINE} @ {WINDOW_S}s, all cohorts: {n_frag} shard(s)")
print(f"  QC columns only  {gb_qc * 1000:9.2f} MB")
print(f"  every column     {gb_all * 1000:9.2f} MB   <- the edges")

In [ ]:
qc = nb.load_dfc(ATLAS_FINE, window_s=WINDOW_S)
print(f"{len(qc):,} windows, {round(nb.mem_mb(qc), 1)} MB in memory")
qc.head()

In [ ]:
# The reliability picture, per cohort. `n_tr_effective` is the column that
# matters: pairwise deletion means two subjects contribute different frame
# counts to the same nominal window.
qc.groupby("cohort").agg(
    windows=("window_id", "size"),
    subs=("sub", "nunique"),
    n_tr_nominal=("n_tr_nominal", "median"),
    n_tr_effective_median=("n_tr_effective", "median"),
    n_tr_effective_min=("n_tr_effective", "min"),
    frac_good=("frac_good_frames", "mean"),
    rank_deficient=("rank_deficient", "mean"),
    crosses_run=("crosses_run_boundary", "mean"),
    crosses_clip=("crosses_clip_boundary", "mean"),
).round(3)

In [ ]:
# Predicted vs. observed rank deficiency at this aperture. A row where the
# prediction says False and the data says >0 means frames were censored down
# past the floor -- which is exactly what heavy motion does.
observed = (qc.groupby("cohort")["rank_deficient"].mean().rename("observed").reset_index())
predicted = grid[(grid["atlas"] == ATLAS_FINE) & (grid["window_s"] == WINDOW_S)][
    ["cohort", "tr", "n_samples", "min_window_s", "expect_rank_deficient"]]
predicted.merge(observed, on="cohort").round(3)

### The same window is not the same measurement in two cohorts

Fisher-z of a correlation over *n* samples has sampling SD `1 / sqrt(n − 3)`.
`n` is `n_tr_effective`, and that depends on the TR and on how many frames
survived censoring — so a 60 s window is a *noisier* estimate in Cam-CAN CC700
(TR 2.47) than in ccfrail (TR 1.12), by roughly the square root of the ratio.

This runs in the same direction as the healthy-vs-frail contrast those two
cohorts are meant to support: more measurement noise in CC700 inflates its
edge variance, which makes the **frail** group look **less variable** than it
is. Censoring makes it worse, not better — ccfrail loses the most frames from
exactly the participants the study is about. The column that lets a model
account for it is `n_tr_effective`, which is why it travels with every row.

In [ ]:
eff = qc.groupby("cohort")["n_tr_effective"].median().rename("n_tr_effective")
noise = pd.concat([cfg["tr"], eff], axis=1).dropna()
noise["nominal_samples"] = (WINDOW_S / noise["tr"]).round().astype(int)
noise["fisher_z_sd"] = 1 / np.sqrt(noise["n_tr_effective"] - 3)
noise["sd_relative_to_best"] = noise["fisher_z_sd"] / noise["fisher_z_sd"].min()
noise.round(3).sort_values("fisher_z_sd")

In [ ]:
if HAVE_MPL:
    cohorts = sorted(qc["cohort"].unique())
    fig, ax = plt.subplots(1, len(cohorts), figsize=(4 * len(cohorts), 3), sharey=True)
    for a, cohort in zip(np.atleast_1d(ax), cohorts):
        sel = qc[qc["cohort"] == cohort]
        a.hist(sel["n_tr_effective"], bins=30)
        a.axvline(sel["n_tr_nominal"].median(), color="k", ls="--", lw=1)
        a.set_title(f"{cohort}\n(dashed = nominal)", fontsize=9)
        a.set_xlabel(f"n_tr_effective @ {WINDOW_S}s")
    np.atleast_1d(ax)[0].set_ylabel("windows")
    fig.tight_layout()

## 3. Edges

Two storage modes, and which one a file uses is a property of the atlas, not of
the cohort: one column per edge named `NodeA__NodeB` below ~20,000 edges, a
packed `fixed_size_list<float32>` in a single `edges` column above it.
`nb.read_subject_edges` (which wraps `dfc.read_edges`) returns the same
`(n_windows, n_edges)` array either way — never branch on the mode yourself.

Edge order is the canonical upper triangle (row-major, `k=1`) of the atlas
label table, and `nb.edge_names` rebuilds it from
`outputs/meta/atlas-<a>_labels.csv`. Read the names from there rather than from
`get_atlas(...)`: the label table is what the data was written against, and it
needs neither nilearn nor network access.

In [ ]:
for atlas, p in n_nodes.items():
    ds, _ = nb.dataset("dfc", atlas)
    cols = nb.edge_columns(ds)
    mode = "list (packed)" if cols == ["edges"] else "columns"
    print(f"{atlas:>15}  {p:>4} nodes  {p * (p - 1) // 2:>6} edges  ->  {mode}")

In [ ]:
# One subject on the fine atlas. Per shard is how edges get touched at 6,105
# columns without materialising a cohort of them.
shards = nb.subject_shards(ATLAS_FINE, WINDOW_S)
print(f"{len(shards)} shard(s) at {ATLAS_FINE} @ {WINDOW_S}s")
qc_one, edges, names = nb.read_subject_edges(shards["path"].iloc[0])
print(f"{shards['cohort'].iloc[0]} / sub-{shards['sub'].iloc[0]}: "
      f"edges {edges.shape}, {edges.nbytes / 1e6:.1f} MB")
print("first edges:", names[:3])
print("NaN edges:", f"{np.isnan(edges).mean():.3%}")

In [ ]:
# Edges are raw r (`fisher_z_applied: false` in the schema metadata), so the
# transform is never already done for you.
z = nb.fisher_z(edges)
window_0 = nb.full_matrix_from_upper(edges[0], len(nb.atlas_labels(ATLAS_FINE)))
print("reconstructed matrix:", window_0.shape,
      "symmetric:", np.allclose(window_0, window_0.T, equal_nan=True))

if HAVE_MPL:
    fig, ax = plt.subplots(1, 2, figsize=(10, 4))
    im = ax[0].imshow(window_0, vmin=-1, vmax=1, cmap="RdBu_r")
    ax[0].set_title(f"{ATLAS_FINE}, window 0", fontsize=9)
    fig.colorbar(im, ax=ax[0], shrink=0.8)
    ax[1].plot(qc_one["start_s"], np.nanstd(z, axis=1))
    ax[1].set_xlabel("window start (s)"); ax[1].set_ylabel("SD of Fisher-z edges")
    ax[1].set_title("edge dispersion over time", fontsize=9)
    fig.tight_layout()

In [ ]:
# A whole cohort of edges, on the coarse atlas where that is affordable.
# guard_gb refuses a read it prices above the guard; the message says how to
# narrow it. Try it first on the fine atlas to see the guard fire.
target = sorted(inv["cohort"].unique())[0]
fine_ds, _ = nb.dataset("dfc", ATLAS_FINE)
fine_filt = ((pads.field("window_s") == str(WINDOW_S)) & (pads.field("cohort") == target))
est, n_frag = nb.estimate_gb(fine_ds, filter=fine_filt)
print(f"{target} @ {ATLAS_FINE} {WINDOW_S}s with edges: {est * 1000:.1f} MB "
      f"across {n_frag} shard(s)")
try:
    nb.load_dfc(ATLAS_FINE, window_s=WINDOW_S, cohort=target,
                with_edges=True, guard_gb=est / 2)
except MemoryError as exc:
    print("\nguard fired, as intended:\n ", exc)

In [ ]:
dfc = nb.load_dfc(ATLAS_COARSE, window_s=WINDOW_S, cohort=target, with_edges=True)
edge_cols = [c for c in dfc.columns if "__" in c]
print(f"{target} @ {ATLAS_COARSE} {WINDOW_S}s: {dfc.shape}, "
      f"{round(nb.mem_mb(dfc), 1)} MB, {len(edge_cols)} edge columns")
dfc[["window_id", "start_s", "n_tr_effective", "rank_deficient"] + edge_cols[:3]].head()

## 4. Who these windows belong to

Same three-file join as stage 2 — curation, pipeline QC, phenotype — reduced to
one row per subject so it can carry a per-subject DFC summary.

`config/phenotype/*_phenotype.csv` does not exist yet: age, sex and the frailty
scores are in no file this pipeline reads or writes, and have to be imported
from each dataset's own source table with `tools/make_phenotype.py`. Until they
are, `has_pheno` is False everywhere and any age- or sex-stratified cell below
will be empty rather than wrong.

In [ ]:
COHORTS = sorted(inv["cohort"].unique())
subjects = pd.concat([nb.subject_table(c) for c in COHORTS], ignore_index=True)
print(subjects.groupby("cohort")["has_pheno"].agg(["size", "sum"])
      .rename(columns={"size": "subject_rows", "sum": "with_phenotype"}))

In [ ]:
# Per-subject DFC summary at one aperture: dispersion of Fisher-z edges over
# windows, which is the simplest thing "dynamic" can mean, plus the reliability
# columns needed to interpret it.
per_sub = (qc.groupby(["cohort", "task", "sub"])
             .agg(n_windows=("window_id", "size"),
                  n_tr_effective_median=("n_tr_effective", "median"),
                  frac_good_frames_dfc=("frac_good_frames", "mean"),
                  frac_rank_deficient=("rank_deficient", "mean"))
             .reset_index())
per_sub["fisher_z_sd_expected"] = 1 / np.sqrt(per_sub["n_tr_effective_median"] - 3)

analysis = subjects.merge(per_sub, on=["cohort", "task", "sub"], how="right")
print(f"{len(analysis)} subject x task row(s) with stage-3 data at "
      f"{ATLAS_FINE} @ {WINDOW_S}s")
analysis.head()

In [ ]:
# What a group comparison would be resting on, before any model. Read the
# rightmost two columns together: they are the acquisition confound, and they
# are not equal across the two Cam-CAN cohorts.
cols = [c for c in ["mean_fd", "frac_good_frames", "frac_good_frames_dfc",
                    "frac_rank_deficient", "n_tr_effective_median",
                    "fisher_z_sd_expected"]
        if c in analysis.columns]
analysis.groupby("cohort")[cols].median().round(3)

## Where this stops

No exclusion applied, no Fisher-z group average, no model. Three things to
carry into whatever comes next:

* **`n_tr_effective`, not `n_tr_nominal`.** Windows differ in how much data
  they actually saw, and the Fisher-z sampling SD is `1/sqrt(n − 3)`.
* **`rank_deficient` rows are computed and flagged, not dropped.** Deciding
  what to do with them is an analysis decision; the pipeline deliberately does
  not make it.
* **`camcan` vs `camcan_ccfrail` is confounded by acquisition.** TR 2.47 vs
  1.12, five echoes vs one — the noise difference points the same way as the
  hypothesis. `config/camcan_ccfrail_movie.yaml`'s header comments have the
  full comparison; read them before running the contrast.

Memory, if you widen any of this:

* QC-only reads are flat in the atlas size — start there, always;
* one dataset object per atlas, and filter on `window_s` / `cohort` / `task` /
  `sub` so directories are pruned before a file opens;
* on `harvardoxford`, edges are per shard via `nb.read_subject_edges`; a pooled
  read with `with_edges=True` at 30 s is 1–2.5 GB for one cohort and grows with
  `1/stride`;
* price it with `nb.estimate_gb(...)` first, and leave `guard_gb` on;
* a full-cohort pass belongs in a batch job that writes a summary table, not in
  this notebook.